In [13]:
!pip install pygame

  Using cached pygame-2.6.1-cp312-cp312-win_amd64.whl.metadata (13 kB)
Using cached pygame-2.6.1-cp312-cp312-win_amd64.whl (10.6 MB)



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from fpdf import FPDF
import os
from datetime import datetime
import random
import cv2
import numpy as np
import time
import pygame
import pandas as pd
import qrcode
from PIL import Image
from deepface import DeepFace
import mediapipe as mp
import smtplib
from email.message import EmailMessage

# Initialize sound system
pygame.mixer.init()
scroll_sound = pygame.mixer.Sound("scroll.mp3")

# Config paths
KNOWN_FACES_DIR = "known_faces"
FOOD_IMAGE_DIR = "food_images"
model_name = "VGG-Face"

# Menu config
menu = {
    "drinks": [("Coke", 2), ("Juice", 3), ("Water", 1), ("Iced Tea", 3), ("Lemonade", 2), ("Coffee", 4), ("Milkshake", 5)],
    "food": [("Burger", 5), ("Pizza", 7), ("Pasta", 6), ("Fries", 3), ("Sandwich", 4), ("Salad", 5), ("Tacos", 6)],
    "desserts": [("Ice Cream", 4), ("Cake", 5), ("Brownie", 3), ("Donut", 2), ("Muffin", 3), ("Pudding", 4), ("Pie", 4)]
}
quantities = {1: "Small", 2: "Medium", 3: "Big"}

# Customer email mapping
customer_emails = {
    "John": "john@example.com",
    "Alice": "alice@example.com",
    "Bob": "bob@example.com",
    "aayush": "aayushkumar93407@gmail.com"
}

# Mediapipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.8)
mp_draw = mp.solutions.drawing_utils

# UI colors
COLORS = {
    "bg": (30, 30, 30),
    "text": (255, 255, 255),
    "highlight": (0, 255, 200),
    "prompt": (255, 180, 80),
    "confirm": (100, 255, 100)
}

# App states
cap = cv2.VideoCapture(0)
current_stage = "detect_face"
selected_name = ""
selected_category = ""
menu_index = 0
selected_item = None
selected_quantity = None
final_bill = 0
face_verified = False
gesture_start_time = time.time()
last_gesture = None
scroll_cooldown = 1.5
last_scroll_time = time.time()

# Email sending function
def send_bill_via_email(name, pdf_path):
    if name not in customer_emails:
        print(f"No email found for {name}, skipping email send.")
        return

    receiver_email = customer_emails[name]
    sender_email = "aayushkumar9340@gmail.com"
    sender_password = ""

    msg = EmailMessage()
    msg['Subject'] = "Your Digital Restaurant Bill"
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg.set_content(f"Hi {name},\n\nThank you for your order! Please find your bill attached.\n\nRegards,\nDigital Restaurant Team")

    with open(pdf_path, 'rb') as f:
        file_data = f.read()

    msg.add_attachment(file_data, maintype='application', subtype='pdf', filename='bill.pdf')

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(sender_email, sender_password)
            smtp.send_message(msg)
        print(f"Email sent successfully to {receiver_email}")
    except Exception as e:
        print(f"Failed to send email: {e}")

# Utility Functions
def draw_text_box(frame, text, y, color="prompt"):
    overlay = frame.copy()
    cv2.rectangle(overlay, (20, y - 30), (620, y + 10), COLORS["bg"], -1)
    alpha = 0.6
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    cv2.putText(frame, text, (40, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLORS[color], 2)

def show_food_image(frame, item):
    img_path = os.path.join(FOOD_IMAGE_DIR, f"{item}.jpg")
    if os.path.exists(img_path):
        food_img = cv2.imread(img_path)
        food_img = cv2.resize(food_img, (160, 160))
        frame[40:200, -200:-40] = food_img
    return frame

def draw_hand_glow(frame, hand_landmarks):
    h, w, _ = frame.shape
    points = []
    for lm in hand_landmarks.landmark:
        cx, cy = int(lm.x * w), int(lm.y * h)
        points.append((cx, cy))
    if points:
        mask = np.zeros((h, w, 3), dtype=np.uint8)
        glow_color = {
            "detect_face": (255, 255, 0),
            "choose_eat_or_drink": (0, 200, 255),
            "menu": (255, 100, 150),
            "confirm_order": (100, 255, 100),
            "quantity": (100, 100, 255),
            "billing": (255, 200, 0),
            "done": (0, 255, 180)
        }.get(current_stage, (0, 200, 255))

        for point in points:
            cv2.circle(mask, point, 20, glow_color, -1)
        blurred = cv2.GaussianBlur(mask, (61, 61), 0)
        frame = cv2.addWeighted(frame, 1, blurred, 0.4, 0)
    return frame

def detect_gesture(hand_landmarks):
    fingers = []
    if hand_landmarks.landmark[4].x < hand_landmarks.landmark[3].x:
        fingers.append(1)
    else:
        fingers.append(0)
    for tip in [8, 12, 16, 20]:
        fingers.append(int(hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y))

    if fingers == [0, 1, 0, 0, 0]: return 1
    elif fingers == [0, 1, 1, 0, 0]: return 2
    elif fingers == [0, 1, 1, 1, 0]: return 3
    elif fingers == [0, 0, 0, 0, 0]: return "fist"
    elif fingers == [1, 1, 1, 1, 1]: return 4
    return None

def detect_double_fist(results):
    count = 0
    for hand in results.multi_hand_landmarks:
        fingers = []
        if hand.landmark[4].x < hand.landmark[3].x:
            fingers.append(1)
        else:
            fingers.append(0)
        for tip in [8, 12, 16, 20]:
            fingers.append(int(hand.landmark[tip].y < hand.landmark[tip - 2].y))
        if fingers == [0, 0, 0, 0, 0]:
            count += 1
    return count == 2

def confirm_gesture(current, detected):
    global gesture_start_time, last_gesture
    if detected == current:
        if time.time() - gesture_start_time >= 1.0:
            return True
    else:
        gesture_start_time = time.time()
        last_gesture = detected
    return False

def generate_qr_code(data):
    qr = qrcode.make(data)
    qr_path = "order_qr.png"
    qr.save(qr_path)
    return qr_path

def generate_pdf_bill(name, item, quantity, amount):
    order_qr_data = f"Name: {name}, Item: {item}, Quantity: {quantity}, Amount: ${amount}"
    order_qr_path = generate_qr_code(order_qr_data)
    payment_qr_path = "payment.jpeg"

    if not os.path.exists(payment_qr_path):
        raise FileNotFoundError("Payment QR code not found at path: your_payment_qr.png")

    bill_no = f"REST{random.randint(1000, 9999)}"
    date_str = datetime.now().strftime("%d %b %Y")

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    pdf.set_font("Arial", 'B', 20)
    pdf.set_text_color(255, 87, 34)
    pdf.cell(200, 10, "Digital Restaurant", ln=True, align='C')

    pdf.set_font("Arial", '', 12)
    pdf.set_text_color(0, 0, 0)
    pdf.cell(200, 10, "kem cho babu moshayyy", ln=True, align='C')
    pdf.ln(5)

    pdf.set_draw_color(100, 100, 100)
    pdf.set_line_width(0.5)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(8)

    pdf.set_font("Arial", '', 12)
    pdf.cell(100, 10, f"Date: {date_str}", ln=False)
    pdf.cell(100, 10, f"Bill No: {bill_no}", ln=True)

    pdf.ln(5)
    pdf.set_font("Arial", 'B', 14)
    pdf.set_text_color(33, 33, 33)
    pdf.cell(200, 10, f"Customer: {name}", ln=True)

    pdf.ln(5)
    pdf.set_font("Arial", 'B', 12)
    pdf.set_fill_color(240, 240, 240)
    pdf.cell(80, 10, "Item", border=1, fill=True)
    pdf.cell(40, 10, "Quantity", border=1, fill=True)
    pdf.cell(70, 10, "Price", border=1, ln=True, fill=True)

    pdf.set_font("Arial", '', 12)
    pdf.cell(80, 10, item, border=1)
    pdf.cell(40, 10, str(quantity), border=1)
    pdf.cell(70, 10, f"${amount}", border=1, ln=True)

    pdf.ln(10)
    pdf.set_font("Arial", 'B', 14)
    pdf.set_text_color(0, 100, 0)
    pdf.cell(200, 10, f"Total Payable: ${amount}", ln=True, align='R')

    pdf.ln(10)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font("Arial", '', 12)
    pdf.cell(200, 10, "Order QR Code (for staff):", ln=True, align='C')
    pdf.image(order_qr_path, x=80, y=pdf.get_y()+5, w=50, h=50)
    pdf.ln(60)

    pdf.set_font("Arial", '', 12)
    pdf.cell(200, 10, "Scan to Pay:", ln=True, align='C')
    pdf.image(payment_qr_path, x=80, y=pdf.get_y()+5, w=50, h=50)

    pdf.ln(60)
    pdf.set_font("Arial", 'I', 10)
    pdf.set_text_color(150, 150, 150)
    pdf.cell(200, 10, "Thank you for dining with us!", ln=True, align='C')

    pdf.output("bill.pdf")

def log_order_to_csv(name, item, quantity, amount):
    order = {
        "Name": name,
        "Item": item,
        "Quantity": quantity,
        "Amount": amount,
        "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    file_exists = os.path.isfile("orders.csv")
    df = pd.DataFrame([order])
    df.to_csv("orders.csv", mode='a', header=not file_exists, index=False)

# Main loop
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb_frame)
    confirmed_gesture = None
    double_fist = False

    if results.multi_hand_landmarks:
        if detect_double_fist(results):
            double_fist = True
        for hand_landmarks in results.multi_hand_landmarks:
            frame = draw_hand_glow(frame, hand_landmarks)
            detected_gesture = detect_gesture(hand_landmarks)
            if detected_gesture and confirm_gesture(last_gesture, detected_gesture):
                confirmed_gesture = detected_gesture
                break

    if current_stage == "detect_face" and not face_verified:
        try:
            result = DeepFace.find(img_path=frame, db_path=KNOWN_FACES_DIR, model_name=model_name, enforce_detection=False)
            if len(result) > 0 and len(result[0]) > 0:
                match = result[0].iloc[0]
                matched_path = match["identity"]
                selected_name = os.path.splitext(os.path.basename(matched_path))[0]
                face_verified = True
                current_stage = "choose_eat_or_drink"
        except Exception as e:
            print("DeepFace error:", e)

    elif current_stage == "choose_eat_or_drink":
        draw_text_box(frame, f"Hi {selected_name}!", 80, "highlight")
        draw_text_box(frame, "1: Drinks | 2: Food | 3: Desserts", 130, "prompt")
        if confirmed_gesture == 1:
            selected_category = "drinks"
            current_stage = "menu"
        elif confirmed_gesture == 2:
            selected_category = "food"
            current_stage = "menu"
        elif confirmed_gesture == 3:
            selected_category = "desserts"
            current_stage = "menu"

    elif current_stage == "menu":
        item, price = menu[selected_category][menu_index]
        draw_text_box(frame, f"Item: {item}", 100, "prompt")
        draw_text_box(frame, f"Price: ${price}", 150, "prompt")
        frame = show_food_image(frame, item)

        if confirmed_gesture == 1 and time.time() - last_scroll_time > scroll_cooldown:
            menu_index = (menu_index + 1) % len(menu[selected_category])
            last_scroll_time = time.time()
            scroll_sound.play()
        elif confirmed_gesture == "fist":
            selected_item = item
            final_bill = price
            current_stage = "confirm_order"

    elif current_stage == "confirm_order":
        draw_text_box(frame, f"You selected: {selected_item}", 100, "highlight")
        draw_text_box(frame, "Show DOUBLE Fist to confirm", 150, "prompt")
        if double_fist:
            current_stage = "quantity"
            time.sleep(0.4)

    elif current_stage == "quantity":
        draw_text_box(frame, "Select Quantity: 1-Small, 2-Medium, 3-Big", 100, "prompt")
        if confirmed_gesture in [1, 2, 3]:
            selected_quantity = quantities[confirmed_gesture]
            current_stage = "billing"

    elif current_stage == "billing":
        draw_text_box(frame, f"{selected_name} ordered {selected_item} ({selected_quantity})", 100, "confirm")
        draw_text_box(frame, f"Total: ${final_bill}", 150, "confirm")
        draw_text_box(frame, "Show Fist to Confirm Order", 200, "prompt")
        if confirmed_gesture == "fist":
            generate_pdf_bill(selected_name, selected_item, selected_quantity, final_bill)
            log_order_to_csv(selected_name, selected_item, selected_quantity, final_bill)
            send_bill_via_email(selected_name, "bill.pdf")
            current_stage = "done"

    elif current_stage == "done":
        draw_text_box(frame, "Order Confirmed! Thank you!", 180, "highlight")
        draw_text_box(frame, "Press 'q' to exit.", 220, "prompt")

    cv2.imshow("Time pass", frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

25-05-04 00:56:16 - Searching [[[125 150 156]
  [142 167 173]
  [142 168 174]
  ...
  [169 194 200]
  [165 192 196]
  [165 192 196]]

 [[139 161 167]
  [142 165 171]
  [141 164 170]
  ...
  [174 199 202]
  [172 198 201]
  [174 202 204]]

 [[127 144 150]
  [128 145 151]
  [140 157 163]
  ...
  [173 198 197]
  [173 200 198]
  [177 205 203]]

 ...

 [[ 55  72  72]
  [ 54  70  70]
  [ 52  67  67]
  ...
  [176 180 181]
  [177 182 183]
  [182 188 189]]

 [[ 55  71  71]
  [ 53  69  69]
  [ 51  67  67]
  ...
  [178 186 186]
  [181 189 189]
  [185 193 193]]

 [[ 57  73  72]
  [ 55  71  70]
  [ 52  68  67]
  ...
  [181 190 190]
  [181 190 190]
  [182 190 190]]] in 1 length datastore
25-05-04 00:56:18 - find function duration 1.9354150295257568 seconds
25-05-04 00:56:18 - Searching [[[184 198 194]
  [183 197 193]
  [183 197 192]
  ...
  [218 230 230]
  [217 229 229]
  [216 228 228]]

 [[183 197 193]
  [183 197 193]
  [183 197 192]
  ...
  [218 230 230]
  [217 229 229]
  [216 228 228]]

 [[183 197